In [1]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [2]:
# =============================================================================
# BLOCK 2: SYNTHESIZED FEATURE ENGINEERING (CORRECTED)
# =============================================================================
print("--- Starting Block 2: Synthesized Feature Engineering ---")
def create_synthesized_features(df_train, df_test):
    # Combine for consistent processing and reset the index
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    # Store the original id for later, as reset_index will remove it
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # --- A) Brute-Force Numerical Interactions ---
    print("Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1','grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] *all_data[NUMS[j]]
    
    # --- B) Date Features ---
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['year'] = all_data['sale_date'].dt.year
    all_data['month'] = all_data['sale_date'].dt.month
    all_data['year_diff'] = all_data['year'] - all_data['year_built']
    
    # --- C) TF-IDF Text Features ---
    print("Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning','join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),max_features=128, binary=True)
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        
        # This concat will now work because both have a simple 0-based index
        all_data = pd.concat([all_data, tfidf_df], axis=1)
    
    # --- D) Log transform some of the new interaction features ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            # Add a small constant to avoid log(0)
            all_data[c] = np.log1p(all_data[c].fillna(0))
    
    # --- E) Final Cleanup ---
    print("Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city','sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    all_data.fillna(0, inplace=True)
    
    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train','sale_price'])
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train','sale_price'])
    
    # Restore the original 'id' as the index
    X.index = train_ids
    X_test.index = test_ids
    X_test = X_test[X.columns]
    return X, X_test

# We need to re-run this from the original dataframes
X, X_test = create_synthesized_features(df_train, df_test)
print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()


--- Starting Block 2: Synthesized Feature Engineering ---
Creating brute-force numerical interaction features...
Creating TF-IDF features for text columns...
Finalizing feature set...

Synthesized FE complete. Total features: 111


10

In [3]:
# =============================================================================
# BLOCK 3: K-FOLD TRAINING OF MEAN MODEL (NO TUNING)
# =============================================================================
print("\n--- STAGE 1: K-Fold Training of Mean Model ---")
print("# Using pre-tuned, optimal hyperparameters.")
# --- YOUR BEST PARAMETERS FOR THE MEAN MODEL ---
# These are the parameters from your most successful Optuna run.
best_params_mean = {
            'eta': 0.041599605162930035,
            'max_depth': 8,
            'subsample': 0.8211034324219306,
            'colsample_bytree': 0.8683430739702909,
            'lambda': 3.717655605557664,
            'alpha': 2.8186169330124836e-05
            }


# --- K-Fold Training ---
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True,random_state=RANDOM_STATE)
oof_mean_preds = np.zeros(len(X))
test_mean_preds = np.zeros(len(X_test))


# THE FIX: Reload the 'grade' column for stratification as the original df was deleted.
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

# Add the other required XGBoost parameters
final_params_mean = {'objective': 'reg:squarederror', 'eval_metric': 'rmse','tree_method': 'hist', 'random_state': RANDOM_STATE, 'n_jobs': -1,**best_params_mean}
for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f" Mean Model - Fold {fold+1}/{N_SPLITS}...")
    model = xgb.XGBRegressor(**final_params_mean, n_estimators=2500,early_stopping_rounds=100)
    model.fit(X.iloc[train_idx], y_true.iloc[train_idx], eval_set=[(X.iloc[val_idx], y_true.iloc[val_idx])], verbose=False)
    oof_mean_preds[val_idx] = model.predict(X.iloc[val_idx])
    test_mean_preds += model.predict(X_test) / N_SPLITS

    
# --- NEW: CALCULATE AND PRINT FINAL OOF RMSE ---
final_mean_rmse = np.sqrt(mean_squared_error(y_true, oof_mean_preds))
print(f"\n# Mean model K-Fold training complete.")
print(f"# Final OOF RMSE for Mean Model: ${final_mean_rmse:,.2f}")
print("-" * 50)



--- STAGE 1: K-Fold Training of Mean Model ---
# Using pre-tuned, optimal hyperparameters.
 Mean Model - Fold 1/5...
 Mean Model - Fold 2/5...
 Mean Model - Fold 3/5...
 Mean Model - Fold 4/5...
 Mean Model - Fold 5/5...

# Mean model K-Fold training complete.
# Final OOF RMSE for Mean Model: $98,990.27
--------------------------------------------------


In [4]:
# =============================================================================
# BLOCK 4: K-FOLD TRAINING OF ERROR MODEL (NO TUNING)
# =============================================================================
print("\n--- STAGE 2: K-Fold Training of Error Model ---")
print("# Using pre-tuned, optimal hyperparameters for the error model.")
# --- YOUR BEST PARAMETERS FOR THE ERROR MODEL ---
# These are the parameters from your most successful error model tuning run.
best_params_error = {
        'eta': 0.01725756977806232,
        'max_depth': 9,
        'subsample': 0.9325133327284854,
        'colsample_bytree': 0.7391175883075835,
        'lambda': 0.5897279418593165,
        'alpha': 0.6057645620141668
        }


# --- K-Fold Training ---
error_target = np.abs(y_true - oof_mean_preds)
X_for_error = X.copy()
X_for_error['mean_pred_oof'] = oof_mean_preds
X_test_for_error = X_test.copy()
X_test_for_error['mean_pred_oof'] = test_mean_preds
oof_error_preds = np.zeros(len(X))
test_error_preds = np.zeros(len(X_test))

grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

# Add the other required XGBoost parameters
final_params_error = {'objective': 'reg:squarederror', 'eval_metric': 'rmse','tree_method': 'hist', 'random_state': RANDOM_STATE, 'n_jobs': -1,**best_params_error}
for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error, grade_for_stratify)):
    print(f" Error Model - Fold {fold+1}/{N_SPLITS}...")
    model = xgb.XGBRegressor(**final_params_error, n_estimators=2000, early_stopping_rounds=100)
    model.fit(X_for_error.iloc[train_idx], error_target.iloc[train_idx], eval_set=[(X_for_error.iloc[val_idx], error_target.iloc[val_idx])], verbose=False)
    oof_error_preds[val_idx] = model.predict(X_for_error.iloc[val_idx])
    test_error_preds += model.predict(X_test_for_error) / N_SPLITS
    
# --- NEW: CALCULATE AND PRINT FINAL OOF RMSE ---
final_error_rmse = np.sqrt(mean_squared_error(error_target, oof_error_preds))
print(f"\n# Error model K-Fold training complete.")
print(f"# Final OOF RMSE for Error Model: ${final_error_rmse:,.2f}")
print("-" * 50)



--- STAGE 2: K-Fold Training of Error Model ---
# Using pre-tuned, optimal hyperparameters for the error model.
 Error Model - Fold 1/5...
 Error Model - Fold 2/5...
 Error Model - Fold 3/5...
 Error Model - Fold 4/5...
 Error Model - Fold 5/5...

# Error model K-Fold training complete.
# Final OOF RMSE for Error Model: $62,782.99
--------------------------------------------------


In [5]:
# =============================================================================
# FINAL ASYMMETRIC CALIBRATION AND SUBMISSION (ULTIMATE ROBUST VERSION)
# =============================================================================
print("\n--- Final Asymmetric Calibration ---")

# --- Safely reload y_true to ensure it's available ---
y_true = pd.read_csv('./dataset.csv')['sale_price']
# --- Your existing correct code ---
oof_error_final = np.clip(oof_error_preds, 0, None)
best_a, best_b, best_metric = 2.0, 2.0, float('inf')
for a in np.arange(1.90, 2.31, 0.01):
    for b in np.arange(2.10, 2.51, 0.01):
        low = oof_mean_preds - oof_error_final * a
        high = oof_mean_preds + oof_error_final * b
        # We need the winkler_score function defined here or in a previous cell
        metric, coverage = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA, return_coverage=True)
        if metric < best_metric:
            best_metric = metric
            best_a, best_b = a, b
print(f"\nGrid search complete. Final OOF Score: {best_metric:,.2f}. Best multipliers: a={best_a:.2f}, b={best_b:.2f}")    

# --- Create Final Submission ---
print("\nCreating final submission file...")
test_error_final = np.clip(test_error_preds, 0, None)
final_lower = test_mean_preds - test_error_final * best_a
final_upper = test_mean_preds + test_error_final * best_b
final_upper = np.maximum(final_lower, final_upper)

# Your excellent, robust fix for the IDs
test_ids = pd.read_csv('./test.csv', usecols=['id'])['id']
submission_df = pd.DataFrame({
    'id': test_ids,
    'pi_lower': final_lower,
    'pi_upper': final_upper
    })

submission_df.to_csv('submission_winner_v1_301.csv', index=False)
print("\n'submission_final_v6.csv' created successfully!")
display(submission_df.head())



--- Final Asymmetric Calibration ---

Grid search complete. Final OOF Score: 301,553.47. Best multipliers: a=1.95, b=2.18

Creating final submission file...

'submission_final_v6.csv' created successfully!


,id,pi_lower,pi_upper
0,200000,832287.494727,1.057812e+06
1,200001,601845.250488,8.275256e+05
2,200002,448866.155176,6.598507e+05
3,200003,281302.292334,4.053173e+05
4,200004,314941.293555,7.683260e+05


In [12]:
import lightgbm as lgb
import optuna

# =======================================================================================
#
# CORRECTED STAGE 3: HYPERPARAMETER TUNING & K-FOLD TRAINING
#
# =======================================================================================

# --- PART A: OPTUNA HYPERPARAMETER SEARCH ---
print("\n--- STAGE 3, PART A: Searching for Optimal Quantile Model Hyperparameters ---")

# Define the objective function for Optuna to minimize the Winkler Score
def create_quantile_objective(X, y_true):
    # Split data just once for the tuning process to make it faster
    X_train, X_val, y_train, y_val = train_test_split(X, y_true, test_size=0.25, random_state=RANDOM_STATE)

    def objective(trial):
        # Define the hyperparameter search space for Optuna to explore
        params = {
            'objective': 'quantile',
            'metric': 'quantile',
            'n_estimators': 2000,
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 100),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
            'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'verbose': -1,
            'n_jobs': -1,
            'seed': RANDOM_STATE
        }

        # Train a model for the lower bound
        lower_params = params.copy()
        lower_params['alpha'] = COMPETITION_ALPHA / 2.0
        model_lower = lgb.LGBMRegressor(**lower_params)
        model_lower.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])

        # Train a model for the upper bound
        upper_params = params.copy()
        upper_params['alpha'] = 1.0 - (COMPETITION_ALPHA / 2.0)
        model_upper = lgb.LGBMRegressor(**upper_params)
        model_upper.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])

        # Calculate the Winkler score on the validation set
        lower_pred = model_lower.predict(X_val)
        upper_pred = model_upper.predict(X_val)
        upper_pred = np.maximum(lower_pred, upper_pred) # Ensure the interval is valid
        
        score = winkler_score(y_val, lower_pred, upper_pred, alpha=COMPETITION_ALPHA)
        return score

    return objective

# Create and run the Optuna study
study = optuna.create_study(direction='minimize')
objective_func = create_quantile_objective(X, y_true)

# THE FIX IS HERE: Use study.optimize() instead of study.run()
study.optimize(objective_func, n_trials=N_OPTUNA_TRIALS)

# --- PART B: K-FOLD TRAINING WITH OPTIMAL PARAMETERS ---
print("\n" + "="*60)
print("             HYPERPARAMETER SEARCH COMPLETE")
print("="*60)
print(f"Best Optuna trial achieved a Winkler Score of: {study.best_value:,.2f}")
print("Best hyperparameters found:")
# Store the best params in a dictionary for later use
best_lgbm_params = study.best_params
# Add the fixed parameters that weren't part of the search
best_lgbm_params['objective'] = 'quantile'
best_lgbm_params['metric'] = 'quantile'
best_lgbm_params['n_estimators'] = 2000
best_lgbm_params['seed'] = RANDOM_STATE
best_lgbm_params['n_jobs'] = -1
best_lgbm_params['verbose'] = -1
print(best_lgbm_params)
print("\n--- STAGE 3, PART B: Starting K-Fold training with these optimal parameters ---")


# Initialize arrays to store predictions
oof_quantile_lower = np.zeros(len(X))
oof_quantile_upper = np.zeros(len(X))
test_quantile_lower = np.zeros(len(X_test))
test_quantile_upper = np.zeros(len(X_test))

# Setup the same K-Fold splits for a fair evaluation
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"\n--- Quantile Model - Fold {fold+1}/{N_SPLITS} ---")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_true.iloc[train_idx], y_true.iloc[val_idx]

    # --- Train Lower Bound Model (5th percentile) ---
    print("Training lower bound model...")
    lower_params = best_lgbm_params.copy()
    lower_params['alpha'] = COMPETITION_ALPHA / 2.0
    model_lower = lgb.LGBMRegressor(**lower_params)
    model_lower.fit(X_train, y_train,
                    eval_set=[(X_val, y_val)],
                    callbacks=[lgb.early_stopping(100, verbose=False)])

    # --- Train Upper Bound Model (95th percentile) ---
    print("Training upper bound model...")
    upper_params = best_lgbm_params.copy()
    upper_params['alpha'] = 1.0 - (COMPETITION_ALPHA / 2.0)
    model_upper = lgb.LGBMRegressor(**upper_params)
    model_upper.fit(X_train, y_train,
                    eval_set=[(X_val, y_val)],
                    callbacks=[lgb.early_stopping(100, verbose=False)])

    # Store OOF and test predictions
    oof_quantile_lower[val_idx] = model_lower.predict(X_val)
    oof_quantile_upper[val_idx] = model_upper.predict(X_val)
    test_quantile_lower += model_lower.predict(X_test) / N_SPLITS
    test_quantile_upper += model_upper.predict(X_test) / N_SPLITS

print("\n\n--- Quantile model K-Fold training complete. ---")
print("--- Proceed to STAGE 4 to evaluate the OOF score of this new tuned model. ---")

[I 2025-07-16 13:56:00,885] A new study created in memory with name: no-name-e7b8342a-878e-4ade-8249-2e226be0af15



--- STAGE 3, PART A: Searching for Optimal Quantile Model Hyperparameters ---


[I 2025-07-16 13:56:21,547] Trial 0 finished with value: 348773.9726836331 and parameters: {'learning_rate': 0.037828044893339734, 'num_leaves': 46, 'feature_fraction': 0.9820313865185624, 'bagging_fraction': 0.8413852574571605, 'bagging_freq': 3, 'lambda_l1': 0.098169152179086, 'lambda_l2': 0.03437970548813572, 'min_child_samples': 12}. Best is trial 0 with value: 348773.9726836331.
[I 2025-07-16 13:56:54,914] Trial 1 finished with value: 347427.52403311816 and parameters: {'learning_rate': 0.03718770385349798, 'num_leaves': 85, 'feature_fraction': 0.7039829187579021, 'bagging_fraction': 0.9232213533614494, 'bagging_freq': 3, 'lambda_l1': 4.9854245168442745e-05, 'lambda_l2': 2.780904648281282e-07, 'min_child_samples': 23}. Best is trial 1 with value: 347427.52403311816.
[I 2025-07-16 13:57:14,737] Trial 2 finished with value: 342224.4930116927 and parameters: {'learning_rate': 0.03800465412867751, 'num_leaves': 92, 'feature_fraction': 0.7059574747743113, 'bagging_fraction': 0.80436705


             HYPERPARAMETER SEARCH COMPLETE
Best Optuna trial achieved a Winkler Score of: 335,289.69
Best hyperparameters found:
{'learning_rate': 0.03311547625317106, 'num_leaves': 33, 'feature_fraction': 0.6765726602135334, 'bagging_fraction': 0.6013035079487907, 'bagging_freq': 4, 'lambda_l1': 2.0697362475901445e-05, 'lambda_l2': 4.904133358285167e-05, 'min_child_samples': 77, 'objective': 'quantile', 'metric': 'quantile', 'n_estimators': 2000, 'seed': 42, 'n_jobs': -1, 'verbose': -1}

--- STAGE 3, PART B: Starting K-Fold training with these optimal parameters ---

--- Quantile Model - Fold 1/5 ---
Training lower bound model...
Training upper bound model...

--- Quantile Model - Fold 2/5 ---
Training lower bound model...
Training upper bound model...

--- Quantile Model - Fold 3/5 ---
Training lower bound model...
Training upper bound model...

--- Quantile Model - Fold 4/5 ---
Training lower bound model...
Training upper bound model...

--- Quantile Model - Fold 5/5 ---
Training 

In [8]:
# --- STAGE 4: Evaluate Quantile Model Performance ---
print("\n--- STAGE 4: Evaluating OOF Performance of Quantile Models ---")

# It's possible for the lower prediction to be higher than the upper.
# We must correct this for a valid interval.
print(f"Correcting {np.sum(oof_quantile_lower > oof_quantile_upper)} inverted OOF intervals.")
oof_quantile_upper = np.maximum(oof_quantile_lower, oof_quantile_upper)

# Calculate the Winkler Score and coverage using the function defined in Block 1
final_quantile_score, final_quantile_coverage = winkler_score(
    y_true,
    oof_quantile_lower,
    oof_quantile_upper,
    alpha=COMPETITION_ALPHA,
    return_coverage=True
)

print("\n" + "="*50)
print("           PERFORMANCE COMPARISON")
print("="*50)
print(f"Two-Stage Model (Mean+Error) OOF Score : 301,553.47")
print(f"Direct Quantile Model OOF Score        : {final_quantile_score:,.2f}")
print("="*50)
print(f"Target Coverage                        : {1-COMPETITION_ALPHA:.2%}")
print(f"Achieved Quantile Model Coverage       : {final_quantile_coverage:.2%}")
print("="*50)

if final_quantile_score < 301553.47:
    print("\nCONCLUSION: The Direct Quantile Model is SUPERIOR. This is a new best score!")
else:
    print("\nCONCLUSION: The Direct Quantile Model did not beat the Two-Stage model.")
    print("This could be due to hyperparameters. Tuning the LightGBM params is recommended.")


--- STAGE 4: Evaluating OOF Performance of Quantile Models ---
Correcting 0 inverted OOF intervals.

           PERFORMANCE COMPARISON
Two-Stage Model (Mean+Error) OOF Score : 301,553.47
Direct Quantile Model OOF Score        : 354,600.22
Target Coverage                        : 90.00%
Achieved Quantile Model Coverage       : 88.32%

CONCLUSION: The Direct Quantile Model did not beat the Two-Stage model.
This could be due to hyperparameters. Tuning the LightGBM params is recommended.


In [10]:
# --- STAGE 4.5: Calibrate OOF Quantile Predictions ---
print("\n--- STAGE 4.5: Calibrating the Quantile Model's Interval Width ---")
print("Our OOF coverage was too low (88.32%), so we will programmatically widen the interval.")

# Calculate the center and the half-width of our current OOF interval
oof_center = (oof_quantile_lower + oof_quantile_upper) / 2
oof_half_width = (oof_quantile_upper - oof_quantile_lower) / 2

best_score = float('inf')
best_factor = 1.0
best_coverage = 0.0

# Search for the best widening factor. We start at 1.0 (no change) and increase.
for factor in np.arange(1.0, 1.25, 0.005):
    # Widen the interval
    calibrated_lower = oof_center - (oof_half_width * factor)
    calibrated_upper = oof_center + (oof_half_width * factor)
    
    score, coverage = winkler_score(y_true, calibrated_lower, calibrated_upper,
                                    alpha=COMPETITION_ALPHA, return_coverage=True)
    
    # Optional: Print the progress
    # print(f"Factor: {factor:.3f}, Score: {score:,.2f}, Coverage: {coverage:.2%}")
    
    if score < best_score:
        best_score = score
        best_factor = factor
        best_coverage = coverage

print("\n" + "="*50)
print("           CALIBRATION RESULTS")
print("="*50)
print(f"Original Uncalibrated Score : {final_quantile_score:,.2f} at {final_quantile_coverage:.2%} coverage.")
print(f"Best Calibrated Score       : {best_score:,.2f} at {best_coverage:.2%} coverage.")
print(f"Optimal Widening Factor     : {best_factor:.3f}")
print("="*50)

if best_score < 301553.47:
    print("\nCONCLUSION: SUCCESS! The CALIBRATED Quantile Model is now the best model.")
else:
    print("\nCONCLUSION: Calibration helped, but still didn't beat the Two-Stage model.")
    print("This suggests the core issue lies in the untuned LightGBM hyperparameters.")


--- STAGE 4.5: Calibrating the Quantile Model's Interval Width ---
Our OOF coverage was too low (88.32%), so we will programmatically widen the interval.

           CALIBRATION RESULTS
Original Uncalibrated Score : 354,600.22 at 88.32% coverage.
Best Calibrated Score       : 354,046.03 at 89.50% coverage.
Optimal Widening Factor     : 1.035

CONCLUSION: Calibration helped, but still didn't beat the Two-Stage model.
This suggests the core issue lies in the untuned LightGBM hyperparameters.


In [9]:
# --- STAGE 5: Create Submission File from Quantile Models ---
print("\n--- STAGE 5: Creating final submission file from Quantile Models ---")

# Correct any inverted intervals in the final test predictions
print(f"Correcting {np.sum(test_quantile_lower > test_quantile_upper)} inverted test intervals.")
test_quantile_upper = np.maximum(test_quantile_lower, test_quantile_upper)

# Use the same robust method for getting test IDs as before
test_ids = pd.read_csv('./test.csv', usecols=['id'])['id']

# Create the submission DataFrame
submission_quantile_df = pd.DataFrame({
    'id': test_ids,
    'pi_lower': test_quantile_lower,
    'pi_upper': test_quantile_upper
})

# Save to a new CSV file
submission_filename = 'submission_quantile_v1.csv'
submission_quantile_df.to_csv(submission_filename, index=False)

print(f"\n'{submission_filename}' created successfully!")
print("Displaying the first 5 rows of the new submission file:")
display(submission_quantile_df.head())


--- STAGE 5: Creating final submission file from Quantile Models ---
Correcting 0 inverted test intervals.

'submission_quantile_v1.csv' created successfully!
Displaying the first 5 rows of the new submission file:


,id,pi_lower,pi_upper
0,200000,846879.610897,1.107858e+06
1,200001,560605.174645,7.813908e+05
2,200002,462014.712287,6.633768e+05
3,200003,321428.444842,4.458479e+05
4,200004,357268.820943,7.259167e+05
